# 0단계: 데이터 준비 (Colab)

| 셀 | 역할 |
|---|---|
| 1 | Google Drive 마운트, 경로 설정 |
| 2 | HoVer train/dev, TF-IDF 검색 결과, 위키 DB(2.16GB) 다운로드. Drive에 한 번만 받고, 이후 세션은 /content로 복사해 씀 |
| 3 | 2-hop만 필터링 |
| 4 | train을 `hpqa_id` 기준으로 train/val 분할. 공식 dev는 최종 테스트로 남김 (공식 test는 라벨 없음) |
| 5 | 위키 DB 구조 확인, 정답 근거 문장이 실제로 꺼내지는지 확인 |
| 5-1 | CoreNLP로 문장을 나눠 정답 근거 문장을 `evidence` 열로 붙임 |
| 6 | 분할 결과 저장 (`evidence` 포함) |


In [21]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/fact-verification'
RAW, PROC = f'{ROOT}/data/raw', f'{ROOT}/data/processed'
os.makedirs(RAW, exist_ok=True); os.makedirs(PROC, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
GH = 'https://raw.githubusercontent.com/hover-nlp/hover/main/data/hover'
UNC = 'https://nlp.cs.unc.edu/data/hover'
files = {
    'hover_train_release_v1.1.json': GH, 'hover_dev_release_v1.1.json': GH,
    'train_tfidf_doc_retrieval_results.json': UNC, 'dev_tfidf_doc_retrieval_results.json': UNC,
    'wiki_wo_links.db': UNC,
}
for name, base in files.items():
    if not os.path.exists(f'{RAW}/{name}'):
        !wget -q --show-progress -O "{RAW}/{name}" "{base}/{name}"

# SQLite는 Drive 위에서 느리므로 로컬 디스크로 복사
WIKI = '/content/wiki_wo_links.db'
if not os.path.exists(WIKI):
    !cp "{RAW}/wiki_wo_links.db" {WIKI}
!ls -lh "{RAW}" 

total 2.2G
-rw------- 1 root root  19M Nov 12  2020 dev_tfidf_doc_retrieval_results.json
-rw------- 1 root root 2.1M Oct 21 05:29 hover_dev_release_v1.1.json
-rw------- 1 root root 8.8M Oct 21 05:29 hover_train_release_v1.1.json
-rw------- 1 root root  84M Nov  3  2020 train_tfidf_doc_retrieval_results.json
-rw------- 1 root root 2.1G Nov  3  2020 wiki_wo_links.db


In [23]:
import json
import pandas as pd

train_all = pd.read_json(f'{RAW}/hover_train_release_v1.1.json')
dev_all = pd.read_json(f'{RAW}/hover_dev_release_v1.1.json')
train2 = train_all[train_all.num_hops == 2].reset_index(drop=True)
test2 = dev_all[dev_all.num_hops == 2].reset_index(drop=True)
print(len(train_all), len(train2), len(dev_all), len(test2))
display(pd.crosstab(train2.num_hops, train2.label, normalize='index'))

18171 9052 4000 1126


label,NOT_SUPPORTED,SUPPORTED
num_hops,,
2,0.282369,0.717631


In [24]:
from sklearn.model_selection import GroupShuffleSplit

# 같은 hpqa_id의 변형 claim이 train/val 양쪽에 들어가지 않도록 그룹 단위로 자름
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
                      .split(train2, groups=train2.hpqa_id))
train, val = train2.iloc[tr_idx].copy(), train2.iloc[va_idx].copy()
assert not set(train.hpqa_id) & set(val.hpqa_id)
for n, d in [('train', train), ('val', val), ('test', test2)]:
    print(n, len(d), d.label.value_counts(normalize=True).round(3).to_dict())

train 8166 {'SUPPORTED': 0.718, 'NOT_SUPPORTED': 0.282}
val 886 {'SUPPORTED': 0.718, 'NOT_SUPPORTED': 0.282}
test 1126 {'NOT_SUPPORTED': 0.537, 'SUPPORTED': 0.463}


In [25]:
import sqlite3
con = sqlite3.connect(WIKI)
print(con.execute("SELECT name, sql FROM sqlite_master WHERE type='table'").fetchall())
print(con.execute('SELECT COUNT(*) FROM documents').fetchone())

# 예시 하나의 정답 문서 원문을 그대로 출력해서 문장 구분 방식을 확인한다
ex = train.iloc[0]
print(ex.claim, ex.label, ex.supporting_facts)
for title in {t for t, _ in ex.supporting_facts}:
    print('---', title)
    print(repr(con.execute('SELECT * FROM documents WHERE id = ?', (title,)).fetchone()))

[('documents', 'CREATE TABLE documents (id PRIMARY KEY, text)')]
(5233329,)
Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and Kristian Zahrtmann. SUPPORTED [['Kristian Zahrtmann', 0], ['Kristian Zahrtmann', 1], ['Peder Severin Krøyer', 1]]
--- Peder Severin Krøyer
('Peder Severin Krøyer', "Peder Severin Krøyer ['peːdə 'seveʁin 'kʁojə] (23 July 1851 – 21 November 1909), professionally known as P. S. Krøyer, was a Danish painter. He is one of the best known and beloved, and the most colorful of the Skagen Painters, a community of Danish and Nordic artists who lived, gathered, or worked in Skagen, Denmark, especially during the final decades of the 19th century. Krøyer was the unofficial leader of the group.")
--- Kristian Zahrtmann
('Kristian Zahrtmann', 'Peder Henrik Kristian Zahrtmann, known as Kristian Zahrtmann, (31 March 1843 – 22 June 1917) was a Danish painter. He was a part of the Danish artistic generation in the late 19th century, al

## 근거 문장 꺼내기

DB `text`에는 문장 구분자가 없다. HoVer 원본 코드는 **Stanford CoreNLP**(`tokenize,ssplit`)로 문장을 나누고, 그 결과의 순서가 `supporting_facts`의 문장 번호와 맞는다. 다른 분리기(nltk 등)를 쓰면 `P. S. Krøyer` 같은 약어에서 번호가 어긋날 수 있으니 원본과 똑같이 CoreNLP를 쓴다.
DB 조회 키는 원본처럼 `unicodedata.normalize('NFD', title)`로 바꾼다.

In [26]:
!which java || apt-get -qq install -y openjdk-17-jre-headless
!pip -q install stanza
import stanza
from stanza.server import CoreNLPClient
import os
# 설치 기본 경로와 서버가 찾는 기본 경로가 달라서 둘을 같은 곳으로 맞춘다
os.environ['CORENLP_HOME'] = '/content/corenlp'
if not os.path.exists('/content/corenlp'):
    stanza.install_corenlp(dir='/content/corenlp')  # ~500MB, 세션마다 1회
# 9000번 포트는 다른 프로세스와 겹칠 수 있어 다른 포트를 쓴다
client = CoreNLPClient(annotators=['tokenize', 'ssplit'], timeout=60000, memory='4G',
                       be_quiet=False, endpoint='http://localhost:9011')
client.start()

/usr/bin/java


INFO:stanza:Writing properties to tmp file: corenlp_server-d54c7c2bd4324fe6.props
INFO:stanza:Starting server with command: java -Xmx4G -cp /content/corenlp/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9011 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet False -serverProperties corenlp_server-d54c7c2bd4324fe6.props -annotators tokenize,ssplit -preload -outputFormat serialized


In [27]:
import unicodedata
from functools import lru_cache

@lru_cache(maxsize=None)
def get_sents(title):
    row = con.execute('SELECT text FROM documents WHERE id = ?',
                      (unicodedata.normalize('NFD', title),)).fetchone()
    if row is None:
        return None
    para = row[0]
    ann = client.annotate(para, output_format='json')
    # 원본과 같이 문자 오프셋으로 원문을 잘라 쓴다
    return tuple(para[s['tokens'][0]['characterOffsetBegin']:s['tokens'][-1]['characterOffsetEnd']]
                 for s in ann['sentences'])

def gold_evidence(sfs):
    out = []
    for title, sid in sfs:
        sents = get_sents(title)
        out.append(sents[sid] if sents is not None and sid < len(sents) else None)
    return out

for d in (train, val, test2):
    d['evidence'] = d.supporting_facts.map(gold_evidence)

# 문서 없음 / 문장 번호 범위 초과가 0에 가까워야 분리 방식이 원본과 맞는 것
for n, d in [('train', train), ('val', val), ('test', test2)]:
    bad = d.evidence.map(lambda ev: None in ev)
    print(n, f'근거 누락 claim {bad.sum()} / {len(d)}')

ex = train.iloc[0]
print(ex.claim, ex.label)
for (t, i), s in zip(ex.supporting_facts, ex.evidence):
    print(f'  [{t}:{i}] {s}')

train 근거 누락 claim 19 / 8166
val 근거 누락 claim 3 / 886
test 근거 누락 claim 2 / 1126
Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and Kristian Zahrtmann. SUPPORTED
  [Kristian Zahrtmann:0] Peder Henrik Kristian Zahrtmann, known as Kristian Zahrtmann, (31 March 1843 – 22 June 1917) was a Danish painter.
  [Kristian Zahrtmann:1] He was a part of the Danish artistic generation in the late 19th century, along with Peder Severin Krøyer and Theodor Esbern Philipsen, who broke away from both the strictures of traditional Academicism and the heritage of the Golden Age of Danish Painting, in favor of naturalism and realism.
  [Peder Severin Krøyer:1] He is one of the best known and beloved, and the most colorful of the Skagen Painters, a community of Danish and Nordic artists who lived, gathered, or worked in Skagen, Denmark, especially during the final decades of the 19th century.


In [28]:
for n, d in [('train', train), ('val', val), ('test', test2)]:
    d.to_json(f'{PROC}/hover2_{n}.jsonl', orient='records', lines=True, force_ascii=False)
!ls -lh "{PROC}" 

total 6.9M
-rw------- 1 root root 768K Oct 21 06:22 hover2_test.jsonl
-rw------- 1 root root 5.5M Oct 21 06:22 hover2_train.jsonl
-rw------- 1 root root 612K Oct 21 06:22 hover2_val.jsonl
